# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# The source file is Windows-1252 encoded, not UTF-8, and the default read
# fails on a curly apostrophe, byte 0x92
df = pd.read_csv('data/AviationData.csv', encoding='cp1252', low_memory=False)
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df.info()

# USState_Codes.csv is supplied with the lab but is not used here

(df.isna().mean() * 100).round(1).sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                88889 non-null  object        
 1   Investigation.Type      88889 non-null  object        
 2   Accident.Number         88889 non-null  object        
 3   Event.Date              88889 non-null  datetime64[ns]
 4   Location                88837 non-null  object        
 5   Country                 88663 non-null  object        
 6   Latitude                34382 non-null  object        
 7   Longitude               34373 non-null  object        
 8   Airport.Code            50132 non-null  object        
 9   Airport.Name            52704 non-null  object        
 10  Injury.Severity         87889 non-null  object        
 11  Aircraft.damage         85695 non-null  object        
 12  Aircraft.Category       32287 non-null  object

Schedule                  85.8
Air.carrier               81.3
FAR.Description           64.0
Aircraft.Category         63.7
Latitude                  61.3
Longitude                 61.3
Airport.Code              43.6
Airport.Name              40.7
Broad.phase.of.flight     30.6
Publication.Date          15.5
Total.Serious.Injuries    14.1
Total.Minor.Injuries      13.4
Total.Fatal.Injuries      12.8
Engine.Type                8.0
Report.Status              7.2
Purpose.of.flight          7.0
Number.of.Engines          6.8
Total.Uninjured            6.7
Weather.Condition          5.1
Aircraft.damage            3.6
Registration.Number        1.6
Injury.Severity            1.1
Country                    0.3
Model                      0.1
Amateur.Built              0.1
Make                       0.1
Location                   0.1
Investigation.Type         0.0
Event.Date                 0.0
Accident.Number            0.0
Event.Id                   0.0
dtype: float64

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Imputing `Aircraft.Category` before filtering**

`Aircraft.Category` is 63.7% empty, and the blanks are not random: the NTSB only began
populating this field consistently around 2008. Filtering on `Aircraft.Category == 'Airplane'`
directly would therefore act as a *date* filter, silently discarding ~99% of pre-2001
accidents — including 18,381 Cessna and 9,997 Piper records that are plainly airplanes.
That would leave the dataset nominally "1983 onwards" while in practice covering only the
last 15 years.

Instead we impute the category from the aircraft itself. Grouping the 32,287 labelled rows by
`Make + Model` gives a lookup of what each airframe actually is. We apply it only where the
mapping is **unambiguous** (`nunique == 1`), so a model label that appears as both an airplane
and a helicopter is left as NaN and excluded rather than guessed.

In [3]:
# --- Impute Aircraft.Category from Make + Model where unambiguous ---
make_model_key = (df['Make'].str.strip().str.upper() + ' '
                  + df['Model'].str.strip().str.upper())

labelled = df['Aircraft.Category'].notna()
category_lut = (
    df[labelled]
      .groupby(make_model_key[labelled])['Aircraft.Category']
      .agg(['nunique', 'first'])
      .query('nunique == 1')['first']      # only airframes with one consistent category
)

missing_before = df['Aircraft.Category'].isna().sum()
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(make_model_key.map(category_lut))
recovered = missing_before - df['Aircraft.Category'].isna().sum()
print(f'Category blanks: {missing_before:,} | recovered by Make+Model lookup: {recovered:,} '
      f'| still unknown: {df["Aircraft.Category"].isna().sum():,}\n')

# --- Filter to the aircraft and events the client cares about ---
steps = {
    'raw': pd.Series(True, index=df.index),
    'event year >= 1983': df['Event.Date'].dt.year >= 1983,
    'not amateur built': df['Amateur.Built'].str.strip().str.lower() == 'no',
    'accidents (not incidents)': df['Investigation.Type'] == 'Accident',
    'airplanes only': df['Aircraft.Category'] == 'Airplane',
}

mask = pd.Series(True, index=df.index)
for label, condition in steps.items():
    mask &= condition
    print(f'{label:<28} {mask.sum():>6,}')

df = df[mask].copy()

print(
    f'\nrows: {len(df):,} | '
    f'years: {df["Event.Date"].dt.year.min()}-'
    f'{df["Event.Date"].dt.year.max()} | '
    f'categories: {df["Aircraft.Category"].unique()}'
)

# Confirm the sample is spread across the whole window, not bunched in recent years
decade_bins = [1982, 1990, 2000, 2008, 2015, 2023]
print('\nrows per era:')
print(df.groupby(pd.cut(df['Event.Date'].dt.year, decade_bins),
                 observed=True).size().to_string())

Category blanks: 56,602 | recovered by Make+Model lookup: 44,360 | still unknown: 12,242

raw                          88,889
event year >= 1983           85,289
not amateur built            76,960
accidents (not incidents)    73,286
airplanes only               58,659

rows: 58,659 | years: 1983-2022 | categories: ['Airplane']

rows per era:
Event.Date
(1982, 1990]    16879
(1990, 2000]    15541
(2000, 2008]    10469
(2008, 2015]     8044
(2015, 2023]     7726


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# --- Injury columns: clean, impute, and derive the key injury measure ---

inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
            'Total.Minor.Injuries', 'Total.Uninjured']

# Injury.Severity encodes the same as text: 'Fatal(N)' gives the death
# count, and 'Non-Fatal'/'Incident'/'Minor'/'Serious' imply zero deaths.
sev = df['Injury.Severity'].astype(str).str.strip()
fatal_from_sev = sev.str.extract(r'Fatal\((\d+)\)')[0].astype(float)
fatal_from_sev[sev.isin(['Non-Fatal', 'Incident', 'Minor', 'Serious'])] = 0
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(fatal_from_sev)

# Assumption: a remaining blank in these tallies means "none in this category"
# Fill with 0 so the counts add up
df[inj_cols] = df[inj_cols].fillna(0)

df['Total.Occupants'] = df[inj_cols].sum(axis=1)

# Fatal or serious injury, as a rate per occupant.
# Deliberately left NaN where every injury tally was blank, so that occupant
# count is 0 and the rate is undefined. This keeps them out of mean injury rates 
# instead of counting them as accidents where nobody was hurt.
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
df['Fatal.Serious.Rate'] = (
    df['Fatal.Serious.Injuries'] / df['Total.Occupants']
).where(df['Total.Occupants'] > 0)

print(f"Accidents with no recorded occupants (rate left NaN): "
      f"{(df['Total.Occupants'] == 0).sum():,}")

df[['Total.Occupants', 'Fatal.Serious.Injuries', 'Fatal.Serious.Rate']].describe()

Accidents with no recorded occupants (rate left NaN): 321


,Total.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Rate
count,58659.000000,58659.000000,58338.000000
mean,4.363934,0.806117,0.277936
std,20.889290,5.249024,0.432172
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,0.800000
max,576.000000,295.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(df['Aircraft.damage'].value_counts(dropna=False))

# Cleaning tasks: strip whitespace, capitalize, replace 'Unknown' with np.nan
df['Aircraft.damage'] = (
    df['Aircraft.damage']
      .str.strip()
      .str.title()
      .replace({'Unknown': np.nan})
)

# Do not drop rows with unrecorded damage. Instead, leave Destroyed as NaN for these rows. 
# Coercing to 0 would assert "not destroyed" when we don't know what happened. 
# As NaN, these rows are skipped by .mean() when computing destruction rates, 
# but we can still use their injury data.
unknown_damage = df['Aircraft.damage'].isna()
print(f"\nRows with unrecorded damage (kept, Destroyed = NaN): {unknown_damage.sum():,}")
print(f"  their mean occupancy: {df.loc[unknown_damage, 'Total.Occupants'].mean():.1f} "
      f"vs {df.loc[~unknown_damage, 'Total.Occupants'].mean():.1f} for the rest")

df['Destroyed'] = np.where(
    unknown_damage, np.nan, df['Aircraft.damage'] == 'Destroyed'
).astype(float)

# Create ordinal encoding
damage_order = ['Minor', 'Substantial', 'Destroyed']
df['Damage.Severity'] = pd.Categorical(
    df['Aircraft.damage'], categories=damage_order, ordered=True
)

print()
print(df['Aircraft.damage'].value_counts(dropna=False))
print(f"\nOverall destruction rate (known-damage rows only): {df['Destroyed'].mean():.1%}")

Aircraft.damage
Substantial    45309
Destroyed      12010
NaN              844
Minor            421
Unknown           75
Name: count, dtype: int64

Rows with unrecorded damage (kept, Destroyed = NaN): 919
  their mean occupancy: 66.3 vs 3.4 for the rest

Aircraft.damage
Substantial    45309
Destroyed      12010
NaN              919
Minor            421
Name: count, dtype: int64

Overall destruction rate (known-damage rows only): 20.8%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks identified:**

1. **Case and whitespace.** `Make` is free text typed by investigators, so `Cessna`, `CESSNA`
   and `cessna ` are all present. Strip and uppercase.
2. **Corporate-suffix fragmentation.** The same manufacturer appears under many legal names:
   `CESSNA` / `CESSNA AIRCRAFT` / `CESSNA AIRCRAFT CO` / `CESSNA AIRCRAFT COMPANY`. Left alone
   this splits one manufacturer's accident record across a dozen keys and destroys the sample
   sizes the client asked us to be careful about. The worst case is Cirrus, where roughly
   two-thirds of accidents sit under `CIRRUS DESIGN CORP` rather than `CIRRUS`.
3. **Successor and joint-build names.** `HAWKER BEECHCRAFT` is Beech; `GRUMMAN AMERICAN` and
   `GRUMMAN-SCHWEIZER` are Grumman airframes; `CESSNA/WEAVER` and `PIPER / LAUDEMAN` are
   modified Cessnas and Pipers. Map these onto the lead manufacturer.
4. **Missing values.** One row has no `Make` and cannot be attributed — drop it.
5. **Rare makes.** Apply the threshold of 50 accidents so that per-make statistics rest on a
   usable sample.

We consolidate with an ordered, first-match-wins list of brand patterns. Order matters:
`AMERICAN CHAMPION` must be tested before `CHAMPION`, and `MCDONNELL DOUGLAS` before
`DOUGLAS`, or the more specific brand would be swallowed by the more general one. Anything not
matching a known brand falls through to a generic suffix stripper, which cleans up the long
tail without needing an entry for every manufacturer.

In [6]:
print(f"Rows: {len(df):,}")
print(f"Missing Make: {df['Make'].isna().sum()} ({df['Make'].isna().mean():.2%})")

# Strip whitespace and make uppercase
df['Make'] = df['Make'].str.strip().str.upper()
print(f"Unique after strip + upper: {df['Make'].nunique():,}")

# Top makes before and after normalizing
print("\nTop 15 normalized:")
print(df['Make'].value_counts().head(15))

# How badly is a single manufacturer fragmented?
for brand in ['CESSNA', 'BOEING', 'PIPER', 'BEECH']:
    variants = sorted(v for v in df['Make'].dropna().unique() if brand in v.upper())
    print(f"\n{brand}: {len(variants)} distinct spellings")
    print(variants[:12])



Rows: 58,659
Missing Make: 2 (0.00%)
Unique after strip + upper: 1,065

Top 15 normalized:
Make
CESSNA            24834
PIPER             13535
BEECH              4412
MOONEY             1178
BOEING             1074
GRUMMAN             968
BELLANCA            915
AIR TRACTOR         630
AERONCA             535
MAULE               495
CHAMPION            474
STINSON             390
LUSCOMBE            368
AERO COMMANDER      329
TAYLORCRAFT         319
Name: count, dtype: int64

CESSNA: 11 distinct spellings
['CESSNA', 'CESSNA AIRCRAFT', 'CESSNA AIRCRAFT CO', 'CESSNA AIRCRAFT CO.', 'CESSNA AIRCRAFT COMPANY', 'CESSNA ECTOR', 'CESSNA REIMS', 'CESSNA SKYHAWK II', 'CESSNA/AIR REPAIR INC', 'CESSNA/WEAVER', 'REIMS-CESSNA']

BOEING: 7 distinct spellings
['BOEING', 'BOEING (STEARMAN)', 'BOEING COMPANY', 'BOEING OF CANADA/DEHAV DIV', 'BOEING STEARMAN', 'BOEING-STEARMAN', 'THE BOEING COMPANY']

PIPER: 15 distinct spellings
['JETPROP DLX PIPER', 'NEW PIPER', 'NEW PIPER AIRCRAFT INC', 'PIPER', 'PIP

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
print(f"Missing Model values: {df['Model'].isna().sum()}")

# Drop NaNs
df = df.dropna(subset=['Model']).copy()

# Normalize case/whitespace
df['Model'] = df['Model'].str.strip().str.upper()

print(f"Unique models after normalizing: {df['Model'].nunique()}")
display(df['Model'].value_counts().head(15))

# Are model labels unique to each make?
makes_per_model = df.groupby('Model')['Make'].nunique().sort_values(ascending=False)
shared = makes_per_model[makes_per_model > 1]
print(f"{len(shared)} of {len(makes_per_model)} model labels appear under more than one Make")
display(shared.head(10))

# Build a combined identifier Make_Model
df['Make_Model'] = df['Make'] + ' ' + df['Model']
print(f"Unique Make_Model plane types: {df['Make_Model'].nunique()}")
display(df['Make_Model'].value_counts().head(15))

Missing Model values: 13
Unique models after normalizing: 3398


Model
152          2216
172          1643
172N         1092
PA-28-140     862
172M          754
150           745
172P          661
182           611
180           594
PA-18-150     556
PA-18         552
150M          549
PA-28-180     546
PA-28-161     523
PA-28-181     508
Name: count, dtype: int64

525 of 3398 model labels appear under more than one Make


Model
CHALLENGER II    14
G-164A           10
G-164B           10
S2R               9
A36               9
AA-5A             8
AA-5B             8
LC41-550FG        7
DRAGONFLY         7
7GCBC             7
Name: Make, dtype: int64

Unique Make_Model plane types: 4243


Make_Model
CESSNA 152         2215
CESSNA 172         1640
CESSNA 172N        1091
PIPER PA-28-140     862
CESSNA 172M         754
CESSNA 150          745
CESSNA 172P         661
CESSNA 182          611
CESSNA 180          594
PIPER PA-18         550
CESSNA 150M         549
PIPER PA-18-150     549
PIPER PA-28-180     546
PIPER PA-28-161     519
PIPER PA-28-181     505
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
              'Purpose.of.flight', 'Broad.phase.of.flight']

print('BEFORE')
for col in other_cols:
    print(f'\n{col}')
    print(df[col].value_counts(dropna=False))

placeholders = ['Unknown', 'Unk', 'UNK', 'Other', 'None', '']

# strip / title-case Engine.Type, Purpose.of.flight, and Broad.phase.of.flight
# replace placeholders with NaN
for col in ['Engine.Type', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    df[col] = df[col].str.strip().str.title().replace(placeholders, np.nan)

# Uppercase the Weather.Condition acronym
df['Weather.Condition'] = (
    df['Weather.Condition'].str.strip().str.upper().replace(placeholders, np.nan)
)

# For Purpose.of.flight, collapse labels that mean the same thing
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race/Show': 'Air Race Show',
    # the four Public Aircraft levels differ only by which government owns the plane
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft',
    # undocumented NTSB codes, setting to NaN
    'Asho': np.nan,
    'Pubs': np.nan,
})

BEFORE

Engine.Type
Engine.Type
Reciprocating    50956
NaN               3664
Turbo Prop        2316
Turbo Fan         1030
Unknown            419
Turbo Jet          243
Turbo Shaft         12
Electric             5
UNK                  1
Name: count, dtype: int64

Weather.Condition
Weather.Condition
VMC    51225
IMC     4493
NaN     2252
UNK      506
Unk      170
Name: count, dtype: int64

Number.of.Engines
Number.of.Engines
1.0    48521
2.0     7154
NaN     2657
3.0      123
4.0       96
0.0       93
8.0        1
6.0        1
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal                     34868
Instructional                 8084
Aerial Application            3278
NaN                           3105
Unknown                       3009
Business                      2723
Positioning                    940
Other Work Use                 520
Ferry                          502
Aerial Observation             385
Executive/corporate            312
Public Aircraft   

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
# Drop columns that are mostly empty or carry no signal

print(f"Shape before: {df.shape}")
na_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
display(na_pct.to_frame('% NaN'))

NA_THRESHOLD = 50
too_sparse = na_pct[na_pct > NA_THRESHOLD].index.tolist()

# One value each so no signal left
constant = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

# Clean row identifiers, report metadata, and geographical data that doesn't
# pertain to the make/model question
admin = ['Event.Id', 'Accident.Number', 'Registration.Number',
         'Report.Status', 'Publication.Date',
         'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name']

to_drop = [c for c in dict.fromkeys(too_sparse + constant + admin) if c in df.columns]
print(f"\nToo sparse (>{NA_THRESHOLD}% NaN): {too_sparse}")
print(f"Constant after filtering:  {constant}")
print(f"Identifiers and metadata:    {[c for c in admin if c in df.columns]}")

df = df.drop(columns=to_drop)

print(f"\nDropped {len(to_drop)} columns. Shape after: {df.shape}")
print("\nRemaining NaN %:")
display((df.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame('% NaN'))


Shape before: (58646, 37)


,% NaN
Schedule,89.2
Air.carrier,82.0
FAR.Description,66.3
Latitude,59.2
Longitude,59.2
Airport.Code,39.2
Airport.Name,36.6
Broad.phase.of.flight,31.7
Publication.Date,16.4
Purpose.of.flight,10.4



Too sparse (>50% NaN): ['Schedule', 'Air.carrier', 'FAR.Description', 'Latitude', 'Longitude']
Constant after filtering:  ['Investigation.Type', 'Aircraft.Category', 'Amateur.Built']
Identifiers and metadata:    ['Event.Id', 'Accident.Number', 'Registration.Number', 'Report.Status', 'Publication.Date', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name']

Dropped 15 columns. Shape after: (58646, 22)

Remaining NaN %:


,% NaN
Broad.phase.of.flight,31.7
Purpose.of.flight,10.4
Engine.Type,7.0
Weather.Condition,5.0
Number.of.Engines,4.5
Aircraft.damage,1.6
Damage.Severity,1.6
Destroyed,1.6
Fatal.Serious.Rate,0.5
Injury.Severity,0.4


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df.to_csv("CleanedAviationData.csv")